# Train the flood photo classifier (MobileNetV2 on Flood10K)

This notebook trains the model that the website's "Report a flood" page uses (`flood-detection/models/flood_model.keras`).

**Before you start**
1. Open this notebook in Google Colab.
2. **Runtime → Change runtime type → T4 GPU.**
3. Create an empty folder named `flood-model` in your Google Drive (`My Drive/flood-model`).
4. Run the cells from the top.

**After a disconnect:** re-run Steps 1–2. Colab wipes `/content`, but your Google Drive is kept (including `best.keras`).

## Step 1: Mount Drive and download Flood10K from Mendeley

Mounts Google Drive (the model is saved there) and downloads the Flood10K dataset (CC BY-NC 4.0) into `/content/raw`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!wget -q --show-progress -O /content/Flood10k.zip "https://data.mendeley.com/public-files/datasets/4sgcd7zdsp/files/d527906d-8c2a-4a1e-9e31-91e19452042a/file_downloaded"
!unzip -q -o /content/Flood10k.zip -d /content/raw -x "__MACOSX/*" "*.DS_Store"
!ls /content/raw/Flood10k

## Step 2: Clean and split 70/15/15

Skips `.svg` files, broken files, exact duplicates and images under 64 px. Every kept image is converted to RGB JPEG
(at most 512 px) and split into `train` / `val` / `test` with a fixed seed. The last line must print `True`
(the `flood-model` folder exists in your Drive).

In [ ]:
import os, random, hashlib
from PIL import Image

SRC = {'flood': '/content/raw/Flood10k/flooded',
       'no_flood': '/content/raw/Flood10k/normal'}
OUT = '/content/flood_data'
SPLIT = (0.70, 0.15, 0.15)
random.seed(42)
seen = set()

def usable(path):
    try:
        with open(path, 'rb') as f:
            h = hashlib.md5(f.read()).hexdigest()
        if h in seen:
            return False
        img = Image.open(path); img.seek(0)
        img = img.convert('RGB')
        if min(img.size) < 64:
            return False
        seen.add(h)
        return True
    except Exception:
        return False

for cls, folder in SRC.items():
    files = sorted(os.path.join(folder, f) for f in os.listdir(folder))
    good = [p for p in files if usable(p)]
    random.shuffle(good)
    a = int(len(good) * SPLIT[0]); b = a + int(len(good) * SPLIT[1])
    for split, paths in {'train': good[:a], 'val': good[a:b], 'test': good[b:]}.items():
        dst = f'{OUT}/{split}/{cls}'; os.makedirs(dst, exist_ok=True)
        for i, p in enumerate(paths):
            img = Image.open(p); img.seek(0); img = img.convert('RGB')
            img.thumbnail((512, 512))
            img.save(f'{dst}/{i:05d}.jpg', quality=92)
    print(f'{cls}: kept {len(good)} of {len(files)} -> train {a}, val {b-a}, test {len(good)-b}')

print(os.path.exists('/content/drive/MyDrive/flood-model'))   # must print True

## Cell A: GPU check

Prints the TensorFlow version and the GPU. Write the version down: the website's venv should use a matching
TensorFlow if the saved model won't load there.

In [ ]:
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices('GPU'))

## Cell B: Load the data

Loads the three splits at 224×224. `flood` is class 1, so the model's output is P(flood).
Class weights balance the two classes during training.

In [ ]:
IMG = (224, 224); BATCH = 32
CLASSES = ['no_flood', 'flood']          # flood = 1, so the output is P(flood)

def load(path, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        path, labels='inferred', label_mode='binary', class_names=CLASSES,
        image_size=IMG, batch_size=BATCH, shuffle=shuffle, seed=42)

train_ds = load('/content/flood_data/train', True).prefetch(tf.data.AUTOTUNE)
val_ds   = load('/content/flood_data/val', False).prefetch(tf.data.AUTOTUNE)
test_ds  = load('/content/flood_data/test', False)

n_flood = len(os.listdir('/content/flood_data/train/flood'))
n_dry   = len(os.listdir('/content/flood_data/train/no_flood'))
class_weight = {0: (n_flood + n_dry) / (2 * n_dry), 1: (n_flood + n_dry) / (2 * n_flood)}
print(class_weight)

## Cell C: Build the model

ImageNet-pretrained MobileNetV2 with a small sigmoid head. Augmentation and the `Rescaling(1/127.5, offset=-1)`
layer are part of the model, so it takes **raw 0–255 pixels**. The website must not scale the pixels itself.

In [ ]:
from tensorflow.keras import layers, models

augment = tf.keras.Sequential([
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.05),
    layers.RandomZoom(0.1), layers.RandomContrast(0.2)], name='augment')

base = tf.keras.applications.MobileNetV2(input_shape=IMG + (3,), include_top=False, weights='imagenet')
base.trainable = False

inputs = layers.Input(shape=IMG + (3,))
x = augment(inputs)
x = layers.Rescaling(1/127.5, offset=-1)(x)     # MobileNetV2 scaling, built into the model
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid', name='flood_prob')(x)
model = models.Model(inputs, outputs)

## Cell D: Training stage 1 (frozen base)

Trains only the new head. Stops early when validation AUC stops improving, and saves the best model
to `My Drive/flood-model/best.keras` as it goes.

In [ ]:
METRICS = ['accuracy', tf.keras.metrics.Precision(name='precision'),
           tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
ckpt = '/content/drive/MyDrive/flood-model/best.keras'
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt, monitor='val_auc', mode='max', save_best_only=True)]

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=METRICS)
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight, callbacks=callbacks)

## Cell E: Training stage 2 (fine-tune the last 30 layers)

Unfreezes the last 30 layers of MobileNetV2 and trains with a much smaller learning rate.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=METRICS)
h2 = model.fit(train_ds, validation_data=val_ds, epochs=10, class_weight=class_weight, callbacks=callbacks)

## Cell F: Learning curve

Train vs. validation AUC across both stages. A large gap would mean overfitting.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(h1.history['auc'] + h2.history['auc'], label='train AUC')
plt.plot(h1.history['val_auc'] + h2.history['val_auc'], label='val AUC')
plt.axvline(len(h1.history['auc']) - 1, ls='--', c='gray', label='fine-tune starts')
plt.legend(); plt.xlabel('epoch'); plt.show()

## Cell G: Cut-off table (validation)

Precision and recall at different cut-offs on the validation set. This is what the agent's Rule 1 cut-offs
(0.80 / 0.50 / 0.30) are based on.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix

def predict_all(ds):
    ys, ps = [], []
    for x, y in ds:
        ps.append(model.predict(x, verbose=0).ravel()); ys.append(y.numpy().ravel())
    return np.concatenate(ys), np.concatenate(ps)

yv, pv = predict_all(val_ds)
print('val AUC', round(roc_auc_score(yv, pv), 3))
print('cut-off  precision  recall')
for t in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    pred = (pv >= t).astype(int)
    print(f'{t:.2f}     {precision_score(yv, pred):.3f}      {recall_score(yv, pred):.3f}')

## Cell H: Final test score (run once)

The held-out test set, scored at 0.50. Run it only once, after you've finished choosing settings, so the
score stays honest.

In [ ]:
yt, pt = predict_all(test_ds)
pred = (pt >= 0.50).astype(int)
print('test AUC', round(roc_auc_score(yt, pt), 3),
      'precision', round(precision_score(yt, pred), 3),
      'recall', round(recall_score(yt, pred), 3))
print(confusion_matrix(yt, pred))

## Cell I: Save to Google Drive

Saves the final model to `My Drive/flood-model/flood_model.keras`. Copy that file to
`flood-detection/models/flood_model.keras` in the project (see `training/README.md`).

In [ ]:
model.save('/content/drive/MyDrive/flood-model/flood_model.keras')
print('saved')